# Horizon runtime vs. #FDs on service_tickets (1M sample)

Companion to `runtime_service_tickets.ipynb`, which fixes the FD set and grows the
row count. Here we do the opposite: **fix the rows at a 1M sample** of
`service_tickets_21m` and grow the number of FDs, from 1 up to all of them, to see
how runtime scales with the FDs (and the distinct attributes they touch).

Because runtime depends on *which* FDs are present (the four §4.1 interaction
cases), for each count `k` we draw **random** subsets rather than a fixed one. Per
`k` we run 4 rounds, each on a fresh random `k`-subset (distinct where the pool
allows), **drop the first round** as cold-cache warmup, and keep the other 3.

Per round we time three stages separately: FD ordering (§5), FD-pattern-graph
build (§3), and repair (§5.2). Note `repair` re-reads the full-width table every
call irrespective of the FD subset, so it carries a roughly constant IO offset;
the FD-count signal lives mostly in `order` + `graph`. Every kept round is written
as one row to a CSV under `experiments/runtime_vs_fds/service_tickets/`.

`service_tickets` has no injected errors or ground truth, so there is **no
precision/recall** here — only runtime. See `README.md` in the experiment folder
for the results writeup.

In [ ]:
import importlib.util
import logging
import random
import sys
import tempfile
import time
from math import comb
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless: render plots to PNG, no GUI backend
import matplotlib.pyplot as plt
import polars as pl

# Same sys.path dance as runtime_service_tickets.ipynb: horizon/ pipeline modules
# use flat imports and need horizon/ on the path; eval uses package imports and
# needs the repo root. horizon/horizon.py is loaded by file path under a private
# name to dodge the package/module clash on `horizon`.
REPO = Path("../..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

spec = importlib.util.spec_from_file_location("horizon_pipeline", HZN / "horizon.py")
pipe = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipe)

from fd_pattern_graph import FDPatternGraph
from static_fd_analysis import get_ordered_fds
from fds.fd import FunctionalDependency
from fds.set_of_fds import SetOfFDs
from horizon.utils.loaders import load_fds

DATASETS = REPO / "datasets"
OUTPUT = REPO / "output"
OUTPUT.mkdir(exist_ok=True)

# Reproducible results (CSV + plot) live in a dedicated experiment folder;
# OUTPUT stays a scratch cache for the reusable row sample.
EXP_DIR = REPO / "experiments" / "runtime_vs_fds" / "service_tickets"
EXP_DIR.mkdir(parents=True, exist_ok=True)

logging.getLogger("horizon").setLevel(logging.ERROR)  # mute per-tuple + ordering logs

In [2]:
# >>> knobs <<<
DATABASE = "service_tickets_21m"
SAMPLE_ROWS = 1_000_000  # fixed row count for every trial
N_ROUNDS = 4             # rounds per FD-count k; round 0 dropped as warmup (3 kept)
SEED = 0                 # reproducible random FD subsets
MAX_ATTEMPTS = N_ROUNDS * 30  # cap resampling when draws are unorderable/duplicate

In [3]:
# Cut the 2M sample ONCE from the head of clean.parquet and reuse it. Parquet (not
# CSV) because every trial re-reads this file dozens of times; the columns come back
# as Utf8 through the pipeline's loaders either way. n_rows makes the scan stop after
# SAMPLE_ROWS, so the full 21M file is never materialised.
ds_dir = DATASETS / DATABASE
clean_parquet = ds_dir / "clean.parquet"
sample_path = OUTPUT / f"{DATABASE}_sample_{SAMPLE_ROWS}.parquet"

if not sample_path.exists():
    pl.scan_parquet(clean_parquet, n_rows=SAMPLE_ROWS).collect().write_parquet(sample_path)
    print(f"{sample_path.name}: {SAMPLE_ROWS:,} rows (written)")
else:
    print(f"{sample_path.name}: {SAMPLE_ROWS:,} rows (reused)")

service_tickets_21m_sample_1000000.parquet: 1,000,000 rows (written)


In [4]:
# Master FD list, captured as plain (lhs_tuple, rhs) pairs. get_ordered_fds and
# FDPatternGraph MUTATE the FD objects and the SetOfFDs in place (fd.order,
# fd.cyclic, set_of_fds.bound_attributes), so every trial must build a FRESH
# SetOfFDs from these pairs — otherwise boundedness/cyclicity leaks between draws.
master = load_fds(ds_dir, sample_path)
master_fds = [(fd.lhs_attributes, fd.rhs) for fd in master]
N_TOTAL = len(master_fds)


def build_set(indices) -> SetOfFDs:
    """Fresh SetOfFDs holding the FDs at the given master-list indices."""
    s = SetOfFDs()
    for i in indices:
        lhs, rhs = master_fds[i]
        s.add_fd(FunctionalDependency(lhs, rhs))
    return s


def fd_label(indices) -> str:
    return "; ".join(f"{','.join(master_fds[i][0])}->{master_fds[i][1]}" for i in indices)


print(f"{N_TOTAL} FDs:")
for i in range(N_TOTAL):
    print(f"  {fd_label([i])}")

10 FDs:
  agency->agency name
  agency name->agency
  community board->borough
  police precinct->borough
  council district->borough
  incident zip->city
  incident zip->borough
  bbl->community board
  bbl->police precinct
  incident address->street name


In [ ]:
# Sweep k = 1..N_TOTAL. For each k draw N_ROUNDS random k-subsets (distinct while
# the pool C(N_TOTAL, k) allows; repeats once exhausted, e.g. k = N_TOTAL). A draw
# whose FDs can't be ordered (get_ordered_fds raises, e.g. an awkward cyclic draw)
# is resampled, not counted. Round 0 of each k is dropped as cold-cache warmup.
rng = random.Random(SEED)
scratch_out = Path(tempfile.gettempdir()) / "horizon_scratch" / f"{DATABASE}_cleaned_scratch.csv"
scratch_out.parent.mkdir(parents=True, exist_ok=True)

rows = []
skipped = []
for k in range(1, N_TOTAL + 1):
    seen = set()
    n_distinct = comb(N_TOTAL, k)
    round_idx = 0
    attempts = 0
    while round_idx < N_ROUNDS and attempts < MAX_ATTEMPTS:
        attempts += 1
        subset = tuple(sorted(rng.sample(range(N_TOTAL), k)))
        # prefer a not-yet-seen subset while distinct ones remain
        if subset in seen and len(seen) < n_distinct:
            continue

        s = build_set(subset)
        t0 = time.perf_counter()
        try:
            ordered_fds = get_ordered_fds(s, DATABASE, OUTPUT, enable_plotting=False)[0]
        except RuntimeError as e:
            skipped.append((k, subset, str(e)))
            continue  # unorderable draw — resample, don't burn a round
        t_order = time.perf_counter() - t0

        seen.add(subset)

        t1 = time.perf_counter()
        graph = FDPatternGraph(sample_path, s)
        t_graph = time.perf_counter() - t1

        t2 = time.perf_counter()
        pipe.repair_dirty_data(
            sample_path, scratch_out, ordered_fds, graph.repair_table, graph, collect_pattern_expressions=False
        )
        t_repair = time.perf_counter() - t2

        warmup = round_idx == 0
        round_idx += 1
        if warmup:
            continue  # drop the first (cold) round

        total = t_order + t_graph + t_repair
        rows.append(
            {
                "n_fds": k,
                "round": round_idx - 1,
                "n_attrs": len(s.unique_attributes),
                "fds": fd_label(subset),
                "order_secs": round(t_order, 3),
                "graph_secs": round(t_graph, 3),
                "repair_secs": round(t_repair, 3),
                "total_secs": round(total, 3),
            }
        )
        print(
            f"k={k:>2}  attrs={len(s.unique_attributes):>2}  round={round_idx - 1}  "
            f"order={t_order:6.3f}s  graph={t_graph:7.3f}s  repair={t_repair:7.3f}s  total={total:7.3f}s"
        )

if skipped:
    print(f"\nresampled {len(skipped)} unorderable draw(s)")
stats = pl.DataFrame(rows)
display(stats)

In [ ]:
# Long format: one row per kept round, both n_fds and n_attrs retained so the
# result can be aggregated either way downstream.
csv_path = EXP_DIR / "results.csv"
stats.write_csv(csv_path)
print(f"wrote {len(stats)} rows to {csv_path}")

In [ ]:
# Runtime vs #attributes (the chosen x-axis). Each kept round is a scatter point;
# the line connects the mean total per distinct attribute count. Saved to EXP_DIR
# and shown inline as a PNG (the Agg backend won't render figures directly).
from IPython.display import Image

agg = (
    stats.group_by("n_attrs")
    .agg(pl.col("total_secs").mean().alias("mean_total"))
    .sort("n_attrs")
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(stats["n_attrs"], stats["total_secs"], c=stats["n_fds"], cmap="viridis", alpha=0.7)
ax.plot(agg["n_attrs"], agg["mean_total"], "k-", lw=1, alpha=0.6, label="mean per #attrs")
sm = plt.cm.ScalarMappable(
    cmap="viridis", norm=plt.Normalize(stats["n_fds"].min(), stats["n_fds"].max())
)
fig.colorbar(sm, ax=ax, label="#FDs")
ax.set_xlabel("# distinct attributes")
ax.set_ylabel("runtime (sec)")
ax.set_title(f"Horizon runtime vs #attributes — {DATABASE} ({SAMPLE_ROWS:,} rows)")
ax.grid(True, ls=":", alpha=0.5)
ax.legend()
fig.tight_layout()

plot_path = EXP_DIR / "runtime_vs_attrs.png"
fig.savefig(plot_path, dpi=120)
plt.close(fig)
Image(str(plot_path))